In [1]:
import pandas as pd
from nlp4bia.datasets.benchmark.medprocner import MedprocnerLoader, MedprocnerGazetteer
from nlp4bia.linking.retrievers import DenseRetriever
from sentence_transformers import SentenceTransformer

model_name = "/gpfs/projects/bsc14/MN4/bsc14/models/entity_linking/procedimiento/biencoder_medprocner_1_epoch_32_batch_5_parents_stag"

st_model = SentenceTransformer(model_name)

df_proc = MedprocnerLoader().df
gaz_proc = MedprocnerGazetteer().df
print(df_proc.shape, gaz_proc.shape)

gaz_proc = gaz_proc.sort_values(by=["code", "mainterm"], ascending=[True, False])
vector_db = st_model.encode(gaz_proc["term"].tolist(), 
                            show_progress_bar=True, 
                            convert_to_tensor=True, 
                            normalize_embeddings=True)

biencoder = DenseRetriever(vector_db=vector_db, model=st_model)

ls_medprocner_train = biencoder.retrieve_top_k(
                                                df_proc["span"].tolist(), 
                                                gaz_proc, 
                                                k=200, 
                                                input_format="text",
                                                return_documents=True
                                            )

/gpfs/projects/bsc14/code/nlp4bia/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(8475, 11) (234674, 5)


Batches: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 265/265 [00:01<00:00, 155.45it/s]


In [2]:
ls_medprocner_train[0]

{'codes': ['13385008',
  '37931006',
  '449264008',
  '68098006',
  '37931006',
  '422834003',
  '386043001',
  '363121006',
  '284024000',
  '23426006',
  '462553003',
  '13385008',
  '846754005',
  '268925001',
  '170608002',
  '301131000',
  '162883003',
  '229493004',
  '162881001',
  '449263002',
  '419922007',
  '449263002',
  '162881001',
  '162885005',
  '162884009',
  '23426006',
  '162882008',
  '385847009',
  '13385008',
  '385847009',
  '165016006',
  '303931008',
  '171255006',
  '241053004',
  '127783003',
  '408867002',
  '165017002',
  '165025000',
  '19086005',
  '171255006',
  '23426006',
  '370801009',
  '410198006',
  '165026004',
  '70115007',
  '20661009',
  '128969008',
  '171255006',
  '303904005',
  '252472004',
  '19086005',
  '57113004',
  '171255006',
  '229295008',
  '169061002',
  '171255006',
  '415570002',
  '363121006',
  '426345008',
  '4541000175105',
  '710777009',
  '252486000',
  '161938003',
  '171255006',
  '170609005',
  '41187008',
  '258058009

In [ ]:
from sentence_transformers import CrossEncoder

ce_model_name = "/gpfs/projects/bsc14/MN4/bsc14/models/entity_linking/procedimiento/crossencoder_medprocner_5_epoch_16_batch"
ce_model = CrossEncoder(ce_model_name, device="cuda")

In [7]:
from tqdm import tqdm

term2code = gaz_proc.set_index("term")["code"].to_dict()
d_reranked_cands = {}
for mention, d_candidates in tqdm(zip(df_proc["span"].tolist(), ls_medprocner_train), total=len(df_proc["span"].tolist())):
    if len(d_candidates["terms"]) > 0:
        ranks = ce_model.rank(mention, d_candidates["terms"], return_documents=True)
        d_reranked_cands["terms"] = [d_rank["text"] for d_rank in ranks]
        d_reranked_cands["codes"] = [term2code[d_rank["text"]] for d_rank in ranks]
        d_reranked_cands["scores"] = [d_rank["score"] for d_rank in ranks]
        d_reranked_cands["mention"] = mention

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 8475/8475 [06:51<00:00, 20.58it/s]


In [8]:
from sentence_transformers import CrossEncoder
from tqdm import tqdm
import numpy as np

ce_model_name = "/gpfs/projects/bsc14/MN4/bsc14/models/entity_linking/procedimiento/crossencoder_medprocner_5_epoch_16_batch"
ce_model = CrossEncoder(ce_model_name, device="cuda")

term2code = gaz_proc.set_index("term")["code"].to_dict()
ls_mentions = df_proc["span"].tolist()
ls_candidates = ls_medprocner_train
# 1) Build one big list of (mention, candidate) pairs, 
#    while keeping track of where each mention’s block starts/ends.
all_pairs  = []   # will hold tuples (mention, candidate_term)
offsets    = []   # will hold (start_idx, end_idx) for each mention
cursor     = 0

for mention, d_candidates in zip(ls_mentions, ls_candidates):
    candidates = d_candidates["terms"]
    if len(candidates) == 0:
        offsets.append((None, None))  # mark empty
        continue

    start = cursor
    # append one pair for each candidate of this mention
    for cand in candidates:
        all_pairs.append((mention, cand))
        cursor += 1
    end = cursor       # exclusive
    offsets.append((start, end))

# 2) Predict all scores in batches
# The CrossEncoder.predict() method returns a single float per pair.
# By default batch_size=32, but you can increase if GPU memory allows (e.g. 128 or 256).
scores = ce_model.predict(all_pairs, batch_size=4096, show_progress_bar=True)

# 3) Now re‐group and sort each mention’s candidates by score

ls_reranked_cands = []
for (mention, d_candidates), (start, end) in zip(zip(df_proc["span"].tolist(), ls_medprocner_train), offsets):
    d_reranked_cands = {}
    
    # slice out this mention’s score vector
    this_scores      = scores[start:end]                 # shape = (num_cands_for_this_mention,)
    this_candidates  = d_candidates["terms"]              # length = same
    this_codes       = [term2code[t] for t in this_candidates]

    # sort by score descending
    sorted_idx = np.argsort(this_scores)[::-1]
    reranked_terms  = [ this_candidates[i] for i in sorted_idx ]
    reranked_codes  = [ this_codes[i]      for i in sorted_idx ]
    reranked_scores = [ float(this_scores[i])   for i in sorted_idx ]

    d_reranked_cands["mention"] = mention
    d_reranked_cands["terms"]   = reranked_terms
    d_reranked_cands["codes"]   = reranked_codes
    d_reranked_cands["scores"]  = reranked_scores
    
    ls_reranked_cands.append(d_reranked_cands)

Batches: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 414/414 [04:44<00:00,  1.46it/s]


In [9]:
len(ls_reranked_cands)

8475

In [ ]:
from nlp4bia.linking.rerankers import CrossEncoderReranker

# Example:
ce_model_path = "/gpfs/projects/bsc14/MN4/bsc14/models/entity_linking/procedimiento/crossencoder_medprocner_5_epoch_16_batch"
reranker = CrossEncoderReranker(
                                    model_path=ce_model_path,
                                    device="cuda",
                                    batch_size=4096,
                                    term2code=term2code,
                                    show_progress_bar=True
                                )

# Run reranking:
ls_reranked = reranker.rerank(ls_mentions[:10], ls_medprocner_train[:10])


Batches: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.13it/s]


Test they are the same

In [30]:
print(ls_reranked[0]["mention"] == ls_reranked_cands[0]["mention"])
print(ls_reranked[0]["terms"] == ls_reranked_cands[0]["terms"])
print(ls_reranked[0]["codes"] == ls_reranked_cands[0]["codes"])
print(np.abs(np.array(ls_reranked[0]["scores"]) - np.array(ls_reranked_cands[0]["scores"])).sum() < 1e-5)

True
True
True
True
